In [0]:
USE CATALOG rearc;
USE SCHEMA gold;

CREATE OR REPLACE TABLE population_statistics AS
SELECT
    AVG(population) AS mean_population,
    STDDEV(population) AS stddev_population
FROM rearc.bronze.population
WHERE year BETWEEN 2013 AND 2018;

num_affected_rows,num_inserted_rows


In [0]:
SELECT * FROM population_statistics;

mean_population,stddev_population
3.22069808E8,4158441.040908095


In [0]:
CREATE OR REPLACE TABLE best_year_per_series AS

WITH yearly_totals AS (

SELECT
    series_id,
    year,
    SUM(value) AS total_value
FROM rearc.silver.productivity
WHERE period IN ('Q01','Q02','Q03','Q04')
GROUP BY series_id, year

),

ranked AS (

SELECT *,
       ROW_NUMBER() OVER(
           PARTITION BY series_id
           ORDER BY total_value DESC, year DESC
       ) rn
FROM yearly_totals

)

SELECT

r.series_id,
r.year AS best_year,
r.total_value,

p.sector_name,
p.measure_text,
p.class_text

FROM ranked r

JOIN (

SELECT DISTINCT
    series_id,
    sector_name,
    measure_text,
    class_text
FROM rearc.silver.productivity

) p

ON r.series_id = p.series_id

WHERE rn = 1

ORDER BY r.series_id;

num_affected_rows,num_inserted_rows


In [0]:
SELECT * FROM best_year_per_series
LIMIT 20;

series_id,best_year,total_value,sector_name,measure_text,class_text
PRS30006011,2022,16.400000000000002,Manufacturing,Employment,All workers
PRS30006012,2022,13.0,Manufacturing,Employment,All workers
PRS30006013,1989,578.369,Manufacturing,Employment,All workers
PRS30006021,2010,14.2,Manufacturing,Average weekly hours,All workers
PRS30006022,2010,8.899999999999999,Manufacturing,Average weekly hours,All workers
PRS30006023,2014,402.51200000000006,Manufacturing,Average weekly hours,All workers
PRS30006031,2022,16.5,Manufacturing,Hours worked,All workers
PRS30006032,1994,14.600000000000001,Manufacturing,Hours worked,All workers
PRS30006033,1989,568.736,Manufacturing,Hours worked,All workers
PRS30006061,1988,28.700000000000003,Manufacturing,Labor compensation,All workers


In [0]:
CREATE OR REPLACE TABLE series_population_history AS

SELECT

p.series_id,
p.year,
p.period,
p.value,
pop.population

FROM rearc.silver.productivity p

LEFT JOIN rearc.bronze.population pop
ON p.year = pop.year

WHERE p.series_id='PRS30006032'
AND p.period='Q01'

ORDER BY p.year;

num_affected_rows,num_inserted_rows


In [0]:
SELECT * FROM series_population_history;

series_id,year,period,value,population
PRS30006032,1988,Q01,2.1,null
PRS30006032,1989,Q01,1.8,null
PRS30006032,1990,Q01,-4.6,null
PRS30006032,1991,Q01,-7.9,null
PRS30006032,1992,Q01,-3.1,null
PRS30006032,1993,Q01,1.3,null
PRS30006032,1994,Q01,1.7,null
PRS30006032,1995,Q01,0.0,null
PRS30006032,1996,Q01,-4.2,null
PRS30006032,1997,Q01,2.8,null


In [0]:
USE CATALOG rearc;
USE SCHEMA gold;

DROP TABLE IF EXISTS population_statistics;
DROP TABLE IF EXISTS best_year_by_series;
DROP TABLE IF EXISTS prs30006032_population;